# Practice - 10 - Unity Catalog privileges, row filters, masks, and ABAC

Prerequisites / execution context: Run 00. Creates a separate security_demo table. SQL grant templates require a real account group and suitable ownership; multi-user verification needs a second identity. Most admin operations are opt-in.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R58: UC privileges reference](https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference); [R59: UC managed tables](https://docs.databricks.com/aws/en/tables/managed); [R60: External tables](https://docs.databricks.com/aws/en/tables/external); [R61: Convert to managed tables](https://docs.databricks.com/aws/en/tables/convert-to-managed); [R62: Table-level row filters and masks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/); [R63: Apply row filters and masks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply); [R64: ABAC overview](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/); [R65: ABAC policy creation](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies); [R66: ABAC DENY policies (Beta)](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/deny-policies); [R67: Unity Catalog lineage](https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-lineage).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Create disposable security data

Create or replace T=SILVER.security_demo with id INT,country STRING,email STRING. Insert (1,IT,one@example.test),(2,FR,two@example.test). Inspect metadata and current grants.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
T = f"{SILVER}.security_demo"

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIEluc3BlY3QgbWV0YWRhdGEKTWFuYWdlZCBhbmQgZXh0ZXJuYWwgYXJlIHN0b3JhZ2UgbGlmZWN5Y2xlIGNob2ljZXM7IGJvdGggY2FuIGJlIGdvdmVybmVkIGJ5IFVDLiBEbyBub3QgY3JlYXRlIGFuIGV4dGVybmFsIHRhYmxlIGluc2lkZSBhIHZvbHVtZSBvciB1c2UgYSBtYW5hZ2VkIHRhYmxlIHBhdGggYXMgYW4gZXh0ZXJuYWwgbG9jYXRpb24uCgoKVGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpcyBzaG93biBuZXh0LiBDb21wYXJlIHlvdXIgZGVzaWduIGFuZCByZXN1bHRzIGJlZm9yZSBjb3B5aW5nIGl0LjwvZGl2PjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlQ9ZiZxdW90O3tTSUxWRVJ9LnNlY3VyaXR5X2RlbW8mcXVvdDsKc3Bhcmsuc3FsKGYmcXVvdDtDUkVBVEUgT1IgUkVQTEFDRSBUQUJMRSB7VH0gKGlkIElOVCxjb3VudHJ5IFNUUklORyxlbWFpbCBTVFJJTkcpIFVTSU5HIERFTFRBJnF1b3Q7KQpzcGFyay5zcWwoZiZxdW90O0lOU0VSVCBJTlRPIHtUfSBWQUxVRVMgKDEsJiN4Mjc7SVQmI3gyNzssJiN4Mjc7b25lQGV4YW1wbGUudGVzdCYjeDI3OyksKDIsJiN4Mjc7RlImI3gyNzssJiN4Mjc7dHdvQGV4YW1wbGUudGVzdCYjeDI3OykmcXVvdDspCnNwYXJrLnNxbChmJnF1b3Q7REVTQ1JJQkUgRVhURU5ERUQge1R9JnF1b3Q7KS5zaG93KHRydW5jYXRlPUZhbHNlKQpzcGFyay5zcWwoZiZxdW90O1NIT1cgR1JBTlRTIE9OIFRBQkxFIHtUfSZxdW90Oykuc2hvdyh0cnVuY2F0ZT1GYWxzZSkKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Coding task 2: Prepare least-privilege SQL

Use existing account group dea_readers. Print SQL granting catalog usage, schema usage and SELECT on T, then revoking table SELECT. Do not execute the grants automatically.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
GROUP = "dea_readers"

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIExlYXN0LXByaXZpbGVnZSBncmFudCBhbmQgcmV2b2tlClNldCBHUk9VUCB0byBhbiBleGlzdGluZyBhY2NvdW50IGdyb3VwLiBTUUwgdGVtcGxhdGVzIGFyZSBwcmludGVkIHNvIHlvdSBjYW4gaW5zcGVjdCB0aGUgc2NvcGUgYmVmb3JlIGFwcGx5aW5nLiBSZXZva2luZyBhIGRpcmVjdCBncmFudCBkb2VzIG5vdCByZW1vdmUgYWNjZXNzIGluaGVyaXRlZCB0aHJvdWdoIGFub3RoZXIgZ3JvdXAgb3IgcGFyZW50IG9iamVjdC4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2RhdGEtZ292ZXJuYW5jZS91bml0eS1jYXRhbG9nL2FjY2Vzcy1jb250cm9sL3ByaXZpbGVnZXMtcmVmZXJlbmNlIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjU4OiBVQyBwcml2aWxlZ2VzIHJlZmVyZW5jZTwvYT4uCgoKVGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpcyBzaG93biBuZXh0LiBDb21wYXJlIHlvdXIgZGVzaWduIGFuZCByZXN1bHRzIGJlZm9yZSBjb3B5aW5nIGl0LjwvZGl2PjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPkdST1VQPSZxdW90O2RlYV9yZWFkZXJzJnF1b3Q7ICAjIGV4aXN0aW5nIGFjY291bnQgZ3JvdXAKcHJpbnQoZiZxdW90O0dSQU5UIFVTRSBDQVRBTE9HIE9OIENBVEFMT0cge0NBVEFMT0d9IFRPIGB7R1JPVVB9YDsmcXVvdDspCnByaW50KGYmcXVvdDtHUkFOVCBVU0UgU0NIRU1BIE9OIFNDSEVNQSB7U0lMVkVSfSBUTyBge0dST1VQfWA7JnF1b3Q7KQpwcmludChmJnF1b3Q7R1JBTlQgU0VMRUNUIE9OIFRBQkxFIHtUfSBUTyBge0dST1VQfWA7JnF1b3Q7KQpwcmludChmJnF1b3Q7UkVWT0tFIFNFTEVDVCBPTiBUQUJMRSB7VH0gRlJPTSBge0dST1VQfWA7JnF1b3Q7KQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 3: Optional: filter rows and mask a column

If finance_team exists and you own T: build SQL UDFs kit_email_mask(STRING) and kit_region_filter(STRING). Finance sees full emails/all countries; other readers see REDACTED and IT only. Bind policies to T. Keep APPLY_POLICIES false until requirements are met.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
APPLY_POLICIES = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIEFwcGx5IHBvbGljaWVzIG9uIHRoZSBkaXNwb3NhYmxlIGRlbW8KT3B0IGluIG9ubHkgaWYgdGhlIGZpbmFuY2VfdGVhbSBhY2NvdW50IGdyb3VwIGV4aXN0cyBhbmQgeW91IG93biB0aGUgZGVtbyB0YWJsZS4gT3RoZXIgdXNlcnMgc2VlIElUIHJvd3Mgb25seSBhbmQgbWFza2VkIGVtYWlsOyBmaW5hbmNlIG1lbWJlcnMgc2VlIGJvdGggY291bnRyaWVzIGFuZCBmdWxsIGVtYWlsLiBUaGUgZnVuY3Rpb24gcGFyYW1ldGVyIHR5cGUgbWF0Y2hlcyB0aGUgYm91bmQgY29sdW1uIHR5cGUuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9kYXRhLWdvdmVybmFuY2UvdW5pdHktY2F0YWxvZy9maWx0ZXJzLWFuZC1tYXNrcy9tYW51YWxseS1hcHBseSIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI2MzogQXBwbHkgcm93IGZpbHRlcnMgYW5kIG1hc2tzPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPkFQUExZX1BPTElDSUVTPUZhbHNlCmlmIEFQUExZX1BPTElDSUVTOgogICAgc3Bhcmsuc3FsKGYmcXVvdDsmcXVvdDsmcXVvdDtDUkVBVEUgT1IgUkVQTEFDRSBGVU5DVElPTiB7U0lMVkVSfS5raXRfZW1haWxfbWFzayh2IFNUUklORykKICAgIFJFVFVSTlMgU1RSSU5HIFJFVFVSTiBDQVNFIFdIRU4gaXNfYWNjb3VudF9ncm91cF9tZW1iZXIoJiN4Mjc7ZmluYW5jZV90ZWFtJiN4Mjc7KSBUSEVOIHYgRUxTRSAmI3gyNztSRURBQ1RFRCYjeDI3OyBFTkQmcXVvdDsmcXVvdDsmcXVvdDspCiAgICBzcGFyay5zcWwoZiZxdW90OyZxdW90OyZxdW90O0NSRUFURSBPUiBSRVBMQUNFIEZVTkNUSU9OIHtTSUxWRVJ9LmtpdF9yZWdpb25fZmlsdGVyKHYgU1RSSU5HKQogICAgUkVUVVJOUyBCT09MRUFOIFJFVFVSTiBpc19hY2NvdW50X2dyb3VwX21lbWJlcigmI3gyNztmaW5hbmNlX3RlYW0mI3gyNzspIE9SIHY9JiN4Mjc7SVQmI3gyNzsmcXVvdDsmcXVvdDsmcXVvdDspCiAgICBzcGFyay5zcWwoZiZxdW90O0FMVEVSIFRBQkxFIHtUfSBBTFRFUiBDT0xVTU4gZW1haWwgU0VUIE1BU0sge1NJTFZFUn0ua2l0X2VtYWlsX21hc2smcXVvdDspCiAgICBzcGFyay5zcWwoZiZxdW90O0FMVEVSIFRBQkxFIHtUfSBTRVQgUk9XIEZJTFRFUiB7U0lMVkVSfS5raXRfcmVnaW9uX2ZpbHRlciBPTiAoY291bnRyeSkmcXVvdDspCiAgICBzcGFyay50YWJsZShUKS5zaG93KCkKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Written / design task 1

Design a two-identity security test. Predict visibility for finance_team members and non-members who can read the table. How do you remove policy bindings and their functions safely?


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNC4gUHJvdmUgYWNjZXNzIHVzaW5nIGlkZW50aXRpZXMKQXMgYW4gYXV0aG9yaXplZCBncm91cCBtZW1iZXIsIHJlYWQgdGhlIGRlbW8uIFJlcGVhdCBhcyBhIG5vbi1tZW1iZXIgd2l0aCB0aGUgcmVxdWlyZWQgdGFibGUvcGFyZW50IHByaXZpbGVnZXMuIFJlY29yZCByb3cgY291bnQgYW5kIGVtYWlsIHZpc2liaWxpdHkuIERvIG5vdCBjbGFpbSBzZWN1cml0eSBpcyB0ZXN0ZWQgYnkgcXVlcnlpbmcgb25jZSBhcyB0aGUgb3duZXIuIFJlbW92ZSB0YWJsZS1sZXZlbCBiaW5kaW5ncyBiZWZvcmUgZHJvcHBpbmcgdGhlaXIgZnVuY3Rpb25zOiBBTFRFUiBUQUJMRSAuLi4gRFJPUCBST1cgRklMVEVSOyBBTFRFUiBUQUJMRSAuLi4gQUxURVIgQ09MVU1OIGVtYWlsIERST1AgTUFTSy4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2RhdGEtZ292ZXJuYW5jZS91bml0eS1jYXRhbG9nL2ZpbHRlcnMtYW5kLW1hc2tzL21hbnVhbGx5LWFwcGx5IiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjYzOiBBcHBseSByb3cgZmlsdGVycyBhbmQgbWFza3M8L2E+Lgo8L2Rpdj48L2RldGFpbHM+')


## Written / design task 2

Design a central ABAC mask on tagged email columns in two disposable tables. Identify required ownership/privileges, tag assignment responsibility and an identity test matrix.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDIgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNS4gQUJBQyBndWlkZWQgZXhlcmNpc2UKSW4gYSBwZXJtaXR0ZWQgZGV2ZWxvcG1lbnQgc2NoZW1hLCBjcmVhdGUgYSBnb3Zlcm5lZCB0YWcgZm9yIHN5bnRoZXRpYyBzZW5zaXRpdmUgY29sdW1ucywgY3JlYXRlIGEgbWFza2luZyBTUUwgVURGLCBhbmQgdXNlIENhdGFsb2cgRXhwbG9yZXIgUG9saWNpZXMgdG8gY3JlYXRlIGEgY29sdW1uLW1hc2sgcG9saWN5IHNjb3BlZCB0byB0YWdnZWQgY29sdW1ucyBmb3IgeW91ciByZWFkZXIgZ3JvdXAuIFVzZSBTaG93IGNvZGUgYW5kIGNvbXBhcmUgaXQgd2l0aCBSNjUuIFRhZyBlbWFpbCBvbiB0d28gbmV3IGRlbW8gdGFibGVzOyB2ZXJpZnkgdGhhdCBib3RoIHJlY2VpdmUgdGhlIHBvbGljeS4gUmVtb3ZlIGEgdGFnIGFuZCByZS1ldmFsdWF0ZSB0aGUgZWZmZWN0aXZlIHBvbGljeS4gUmVjb3JkIHdobyBjYW4gYXNzaWduIHRoZSB0YWcgYW5kIG1vZGlmeSB0aGUgcG9saWN5LiBLZWVwIHRoaXMgc2VwYXJhdGUgZnJvbSB0aGUgbWFudWFsbHkgbWFza2VkIHRhYmxlIHRvIGF2b2lkIG92ZXJsYXBwaW5nLXBvbGljeSBhbWJpZ3VpdHkuIElmIHRhZy9wb2xpY3kgYWRtaW5pc3RyYXRpb24gaXMgdW5hdmFpbGFibGUsIHN1Ym1pdCB0aGUgZGVzaWduIGFuZCB0ZXN0IG1hdHJpeCBhcyBhIHBhcGVyIGV4ZXJjaXNlLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vZGF0YS1nb3Zlcm5hbmNlL3VuaXR5LWNhdGFsb2cvYWJhYy8iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNjQ6IEFCQUMgb3ZlcnZpZXc8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2RhdGEtZ292ZXJuYW5jZS91bml0eS1jYXRhbG9nL2FiYWMvcG9saWNpZXMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNjU6IEFCQUMgcG9saWN5IGNyZWF0aW9uPC9hPi4KPC9kaXY+PC9kZXRhaWxzPg==')


## Written / design task 3

What does current documented ABAC DENY support? Can it replace a generic DENY SELECT or a column mask? Check current release/compute requirements.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDMgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNi4gREVOWSBkaXN0aW5jdGlvbgpUaGUgZXhhbSBvdXRsaW5lIG1lbnRpb25zIERFTlkuIEN1cnJlbnQgb2ZmaWNpYWwgQUJBQyBERU5ZIGlzIEJldGEsIHJlc3RyaWN0cyBNQU5BR0UgQUNDRVNTIENPTlRST0wsIGFuZCBoYXMgaXRzIG93biByZXF1aXJlbWVudHMuIERvIG5vdCB0cmFuc2xhdGUgaXQgaW50byBhIGdlbmVyaWMgREVOWSBTRUxFQ1QgY29tbWFuZCBvciB1c2UgaXQgYXMgY29sdW1uIG1hc2tpbmcuIFJlYWQgdGhlIGN1cnJlbnQgc3VwcG9ydGVkIHByaXZpbGVnZSBsaXN0IGJlZm9yZSBhcHBseWluZy4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2RhdGEtZ292ZXJuYW5jZS91bml0eS1jYXRhbG9nL2FiYWMvZGVueS1wb2xpY2llcyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI2NjogQUJBQyBERU5ZIHBvbGljaWVzIChCZXRhKTwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')


## Written / design task 4

Explain an eligible external-to-managed Delta conversion and its rollback window. Is UNSET MANAGED a general conversion command for every managed table? Describe how to inspect lineage.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDQgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNy4gQ29udmVyc2lvbiBhbmQgbGluZWFnZQpPcHRpb25hbCBhZG1pbiBleGVyY2lzZTogb24gYSBkaXNwb3NhYmxlIGV4dGVybmFsIERlbHRhIHRhYmxlIHVuZGVyIGFuIGF1dGhvcml6ZWQgZXh0ZXJuYWwgbG9jYXRpb24sIGV4ZWN1dGUgQUxURVIgVEFCTEUgLi4uIFNFVCBNQU5BR0VELCBpbnNwZWN0IGl0cyB0eXBlIGFuZCBsb2NhdGlvbiwgdGhlbiBBTFRFUiBUQUJMRSAuLi4gVU5TRVQgTUFOQUdFRCB3aXRoaW4gdGhlIGRvY3VtZW50ZWQgMTQtZGF5IHJvbGxiYWNrIHdpbmRvdy4gVGhpcyBpcyByb2xsYmFjayBvZiB0aGF0IGNvbnZlcnNpb24sIG5vdCBhbiBhcmJpdHJhcnkgbWFuYWdlZC10by1leHRlcm5hbCBjb252ZXJzaW9uLiBVc2UgUjYxIGZvciBydW50aW1lIGFuZCBhY2Nlc3MgcmVxdWlyZW1lbnRzLiBGaW5hbGx5IGluc3BlY3QgdGhlIG1haW4gR29sZCB0YWJsZSBsaW5lYWdlIGluIENhdGFsb2cgRXhwbG9yZXIuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi90YWJsZXMvY29udmVydC10by1tYW5hZ2VkIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjYxOiBDb252ZXJ0IHRvIG1hbmFnZWQgdGFibGVzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9kYXRhLWdvdmVybmFuY2UvdW5pdHktY2F0YWxvZy9kYXRhLWxpbmVhZ2UiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNjc6IFVuaXR5IENhdGFsb2cgbGluZWFnZTwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')
